## 1. import libraries and configure the project

Before collecting data, I import the Python libraries needed throughout this notebook.

- `requests` is used to send requests to the SEC EDGAR API.
- `json` is used to work with the JSON responses returned by the API.
- `time` is used to add short pauses between requests to avoid sending too many requests too quickly.
- `os` is used to create and manage folders where the raw data will be stored.

I also define the location where the raw API responses will be saved (`data/raw/`) and create a `User-Agent` header, which the SEC requires when accessing its API.

In [3]:
import requests
import json
import time

RAW_DIR = "../data/raw"

HEADERS = {"User-Agent": "ME204 Final Project - S.E.Yzusqui@lse.ac.uk"}

## 2. Retrieve the company ticker database

The SEC identifies each company using a unique **Central Index Key (CIK)**. Before requesting the data, I fetch the SEC's company ticker database.

The company ticker mapping was obtained from the SEC's official site sec.gov. This file links company tickers to their CIK, which is required to find companies financial data.

I inspect the first 3 entries to view the structure of the data and confirm that the request was successful. This information will be used later to obtain the correct CIK for each company included in the analysis.

In [4]:
url = "https://www.sec.gov/files/company_tickers.json"
response = requests.get(url, headers=HEADERS)
ticker_data = response.json()


get_cik = {}

for entry in ticker_data.values():

    ticker = entry["ticker"]

    cik = str(entry["cik_str"]).zfill(10)

    get_cik[ticker] = cik

get_cik["XOM"] = "0000034088"


In [5]:
list(ticker_data.items())[:3]

[('0', {'cik_str': 1045810, 'ticker': 'NVDA', 'title': 'NVIDIA CORP'}),
 ('1', {'cik_str': 320193, 'ticker': 'AAPL', 'title': 'Apple Inc.'}),
 ('2', {'cik_str': 1652044, 'ticker': 'GOOGL', 'title': 'Alphabet Inc.'})]

## 3. Select companies 

The project analyses three sectors: Technology, Healthcare, and Energy. I select three companies from each sector to compare.

Using the company ticker database collected previously, I create a mapping between each company's ticker symbol and its CIK number.

In [6]:
companies = {
    "MSFT": {"name": "Microsoft", "sector": "Technology"},
    "NVDA": {"name": "Nvidia", "sector": "Technology"},
    "AAPL": {"name": "Apple", "sector": "Technology"},
    "PFE": {"name": "Pfizer", "sector": "Healthcare"},
    "JNJ": {"name": "Johnson & Johnson", "sector": "Healthcare"},
    "SYK": {"name": "Stryker", "sector": "Healthcare"},
    "XOM": {"name": "ExxonMobil", "sector": "Energy"},
    "CVX": {"name": "Chevron", "sector": "Energy"},
    "DUK": {"name": "Duke Energy", "sector": "Energy"},
}

for ticker in companies:

    name = companies[ticker]["name"]
    sector = companies[ticker]["sector"]
    cik = get_cik.get(ticker)

    print(ticker, "|", name, "|", sector, "|", cik)

MSFT | Microsoft | Technology | 0000789019
NVDA | Nvidia | Technology | 0001045810
AAPL | Apple | Technology | 0000320193
PFE | Pfizer | Healthcare | 0000078003
JNJ | Johnson & Johnson | Healthcare | 0000200406
SYK | Stryker | Healthcare | 0000310764
XOM | ExxonMobil | Energy | 0000034088
CVX | Chevron | Energy | 0000093410
DUK | Duke Energy | Energy | 0001326160


In [7]:
for ticker in companies:
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{get_cik[ticker]}.json"
    facts = requests.get(url, headers=HEADERS).json()
    gaap = facts["facts"]["us-gaap"]

    revenue_tags = [t for t in gaap if "Revenue" in t and "Cost" not in t and "Deferred" not in t]
    print(f"\n{ticker}:")
    for t in revenue_tags[:10]:
        print("   ", t)

    time.sleep(1)


MSFT:
    BusinessAcquisitionsProFormaRevenue
    ContractWithCustomerLiabilityRevenueRecognized
    EntityWideDisclosureOnGeographicAreasRevenueFromExternalCustomersAttributedToEntitysCountryOfDomicile
    EntityWideDisclosureOnGeographicAreasRevenueFromExternalCustomersAttributedToForeignCountries
    RevenueFromContractWithCustomerExcludingAssessedTax
    RevenueRemainingPerformanceObligation
    RevenueRemainingPerformanceObligationPercentage
    Revenues
    SalesRevenueGoodsNet
    SalesRevenueNet

NVDA:
    BusinessAcquisitionProFormaRevenue
    BusinessAcquisitionsProFormaRevenue
    ContractWithCustomerLiabilityRevenueRecognized
    RevenueFromContractWithCustomerExcludingAssessedTax
    RevenueRemainingPerformanceObligation
    RevenueRemainingPerformanceObligationPercentage
    Revenues

AAPL:
    ContractWithCustomerLiabilityRevenueRecognized
    RevenueFromContractWithCustomerExcludingAssessedTax
    Revenues
    SalesRevenueNet
    SalesRevenueServicesGross

PFE:
    Dis

## 4. Collect financial data from the SEC API

The SEC provides financial information through its XBRL Company Concepts API. Each financial measure is identified using a specific tag, such as revenue, research and development expenses, and operating cash flow.

For this project, I collect five financial concepts for each selected company:

- **Revenues**: measures company sales and is used to normalise comparisons between companies of different sizes.
- **ResearchAndDevelopmentExpense**: measures investment in innovation and future product development.
- **NetIncomeLoss**: measures the final profit remaining after expenses, interest, and taxes.
- **NetCashProvidedByUsedInOperatingActivities**: measures cash generated from normal business operations.
- **PaymentsToAcquirePropertyPlantAndEquipment**: measures capital expenditure, representing investment in physical assets needed for future growth.

The same financial concept is not always reported under exactly the same SEC tag by every company. Therefore, alternative tags are defined for companies where the standard tag is unavailable. This ensures that the data collection process remains consistent while accounting for differences in company reporting practices.

The original API responses are saved in the `data/raw/` folder to preserve the raw data collected from the SEC.

In [8]:
tags = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "RevenueFromContractWithCustomerIncludingAssessedTax",
    "ResearchAndDevelopmentExpense",
    "NetIncomeLoss",
    "NetCashProvidedByUsedInOperatingActivities",
    "PaymentsToAcquirePropertyPlantAndEquipment"
]

for ticker in companies:
    cik = get_cik[ticker]

    for tag in tags:

        alternative_tag = tag
        if ticker in ("PFE", "JNJ") and tag == "ResearchAndDevelopmentExpense":
            alternative_tag = "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost"
        if ticker in ("CVX", "NVDA") and tag == "PaymentsToAcquirePropertyPlantAndEquipment":
            alternative_tag = "PaymentsToAcquireProductiveAssets"

        url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{alternative_tag}.json"
        response = requests.get(url, headers=HEADERS)

        if response.status_code == 200:
            filename = f"{RAW_DIR}/{ticker}_{tag}.json"
            with open(filename, "w") as f:
                json.dump(response.json(), f)
            print(f"Saved: {ticker} - {tag}")
        else:
            print(f"MISSING: {ticker} - {tag} (status {response.status_code})")

        time.sleep(0.2)

Saved: MSFT - Revenues
Saved: MSFT - RevenueFromContractWithCustomerExcludingAssessedTax
MISSING: MSFT - RevenueFromContractWithCustomerIncludingAssessedTax (status 404)
Saved: MSFT - ResearchAndDevelopmentExpense
Saved: MSFT - NetIncomeLoss
Saved: MSFT - NetCashProvidedByUsedInOperatingActivities
Saved: MSFT - PaymentsToAcquirePropertyPlantAndEquipment
Saved: NVDA - Revenues
Saved: NVDA - RevenueFromContractWithCustomerExcludingAssessedTax
MISSING: NVDA - RevenueFromContractWithCustomerIncludingAssessedTax (status 404)
Saved: NVDA - ResearchAndDevelopmentExpense
Saved: NVDA - NetIncomeLoss
Saved: NVDA - NetCashProvidedByUsedInOperatingActivities
Saved: NVDA - PaymentsToAcquirePropertyPlantAndEquipment
Saved: AAPL - Revenues
Saved: AAPL - RevenueFromContractWithCustomerExcludingAssessedTax
MISSING: AAPL - RevenueFromContractWithCustomerIncludingAssessedTax (status 404)
Saved: AAPL - ResearchAndDevelopmentExpense
Saved: AAPL - NetIncomeLoss
Saved: AAPL - NetCashProvidedByUsedInOperating

## 5. Check availability of financial concepts

Before analysing the data, I check whether each selected company reports the required financial concepts in the SEC XBRL database.

Companies do not always use identical XBRL tags for the same financial measure. This step identifies missing concepts so that alternative tags can be considered before extracting the final dataset.

This will help to ensure that the measures used in the analysis are useful to compare across companies.

In [ ]:
check = ["MSFT","NVDA","AAPL","PFE","JNJ","SYK","XOM","CVX","DUK"]


for tkr in check:

    if tkr == "XOM":
        cik = "0000034088"
    else:
        cik = get_cik[tkr]

    d = requests.get(
        f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json",
        headers=HEADERS
    ).json()

    # Keep only US accounting concepts
    gaap = d["facts"].get("us-gaap", {})


    missing_tags = []

    for tag in tags:
        if tag not in gaap:
            missing_tags.append(tag)


    print(f"{tkr:5} {d['entityName'][:26]:28} missing: {missing_tags}")


    # Wait before next request
    time.sleep(1)

MSFT  MICROSOFT CORPORATION        missing: ['RevenueFromContractWithCustomerIncludingAssessedTax']
NVDA  NVIDIA CORP                  missing: ['RevenueFromContractWithCustomerIncludingAssessedTax']
AAPL  Apple Inc.                   missing: ['RevenueFromContractWithCustomerIncludingAssessedTax']
PFE   Pfizer Inc.                  missing: ['RevenueFromContractWithCustomerIncludingAssessedTax', 'ResearchAndDevelopmentExpense']
JNJ   Johnson & Johnson            missing: ['RevenueFromContractWithCustomerIncludingAssessedTax']
SYK   STRYKER CORP                 missing: ['RevenueFromContractWithCustomerExcludingAssessedTax', 'RevenueFromContractWithCustomerIncludingAssessedTax']
XOM   Exxon Mobil Corporation      missing: ['RevenueFromContractWithCustomerIncludingAssessedTax']
CVX   Chevron Corp                 missing: ['PaymentsToAcquirePropertyPlantAndEquipment']
DUK   DUKE ENERGY CORPORATION      missing: ['ResearchAndDevelopmentExpense']


## 6. Find alternative tags for missing concepts

Some companies report similar financial information using different tags. To solve and clean the data collection, I search the available SEC concepts for companies where the standard tag is missing.

This helps to identify alternative tags that has the same financial concept and improves consistency across the selected companies.

In [10]:
cik = get_cik["PFE"]
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
response = requests.get(url, headers=HEADERS)
facts = response.json()

all_tags = facts["facts"]["us-gaap"].keys()

matches = [tag for tag in all_tags if "Research" in tag]
matches

['EffectiveIncomeTaxRateReconciliationNondeductibleExpenseResearchAndDevelopment',
 'EffectiveIncomeTaxRateReconciliationTaxCreditsResearch',
 'ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost',
 'ResearchAndDevelopmentInProcess',
 'ResearchAndDevelopmentAssetAcquiredOtherThanThroughBusinessCombinationWrittenOff',
 'DeferredTaxAssetsInProcessResearchAndDevelopment',
 'IncomeTaxReconciliationTaxCreditsResearch']

In [11]:
# Nvidia's capex and J&J's earlier R&D are filed under different names.
for ticker, keyword in [("NVDA", "Payments"), ("JNJ", "ResearchAndDevelopment")]:
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{get_cik[ticker]}.json"
    facts = requests.get(url, headers=HEADERS).json()

    print(f"\n{ticker}:")
    for tag in facts["facts"]["us-gaap"]:
        if keyword in tag:
            print("   ", tag)

    time.sleep(1)


NVDA:
    CapitalLeasesFutureMinimumPaymentsDue
    CapitalLeasesFutureMinimumPaymentsDueCurrent
    CapitalLeasesFutureMinimumPaymentsDueInFiveYears
    CapitalLeasesFutureMinimumPaymentsDueInFourYears
    CapitalLeasesFutureMinimumPaymentsDueInThreeYears
    CapitalLeasesFutureMinimumPaymentsDueInTwoYears
    CapitalLeasesFutureMinimumPaymentsDueThereafter
    CapitalLeasesFutureMinimumPaymentsPresentValueOfNetMinimumPayments
    LesseeOperatingLeaseLiabilityPaymentsDue
    LesseeOperatingLeaseLiabilityPaymentsDueAfterYearFive
    LesseeOperatingLeaseLiabilityPaymentsDueNextTwelveMonths
    LesseeOperatingLeaseLiabilityPaymentsDueYearFive
    LesseeOperatingLeaseLiabilityPaymentsDueYearFour
    LesseeOperatingLeaseLiabilityPaymentsDueYearThree
    LesseeOperatingLeaseLiabilityPaymentsDueYearTwo
    LesseeOperatingLeaseLiabilityPaymentsRemainderOfFiscalYear
    OperatingLeasePayments
    OperatingLeasesFutureMinimumPaymentsDue
    OperatingLeasesFutureMinimumPaymentsDueCurrent
    Op